# Extended Kalman Filter, Fit with Multiple Time-Series

In [ ]:
import pandas as pd
import numpy as np

import numpyro
numpyro.set_host_device_count(4)
import jax.numpy as jnp

from numpyro import distributions as dist
from numpyro.infer import MCMC, NUTS

import matplotlib.pyplot as plt
from jax import random

from wunkui.models.ssp.kalman_1d import (
    kalman_filter_1d_ekf,
)

In [ ]:
df = pd.read_csv(
    '../resource/sparse/sparse_df.csv', parse_dates=['date']
)
regressors = [
    "ch_a1",
    "ch_b1",
    "ch_b2",
    "ch_a2",
    "ch_b3",
    "ch_b4",
    "ch_a3",
    "ch_a4",
    "ch_b5",
    "ch_b6",
    "ch_b7",
    "ch_b8",
    "ch_a5",
]
coefs_df = None

# df = df.groupby("date")[["outcome"] + regressors].sum().reset_index()

response = df["outcome"].values
response_mean = response.mean()
response_std = response.std()

y = (response - response_mean) / response_std
y = jnp.array(y)

X = df[regressors].values
X = (X - X.mean(axis=0)) / X.std(axis=0)
Z = jnp.concatenate([jnp.ones((X.shape[0], 1)), jnp.array(X)], -1)

positivity_idx = jnp.array([0] + [1] * len(regressors))

print("y shape", y.shape)
print("Z shape", Z.shape)
print("X shape", X.shape)

In [ ]:
dmas = df["dma"].unique()

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
for dma in dmas:
    sub = df[df["dma"] == dma].sort_values("date")
    response = sub["outcome"].values
    offset = sub["trend"].values + sub["weekly"].values + sub["yearly"].values
    residuals = response - offset
    res_std = np.std(residuals)
    normalized_res = residuals / res_std
    axes[0].plot(sub["date"], response, label=dma, alpha=0.7)
    axes[1].plot(sub["date"], offset, label=dma, alpha=0.7)
    axes[2].plot(sub["date"], normalized_res, label=dma, alpha=0.7)

axes[0].set_title("response")
axes[1].set_title("offset")
axes[2].set_title("normalized residuals")
axes[2].set_xlabel("Date")
for ax in axes:
    ax.legend(fontsize=7, ncol=4, loc="best")
plt.tight_layout()

In [ ]:
# EKF initialization — mixed: intercept is linear, channel states use exp(0.5·a)
#
# positivity_idx: False for intercept (linear, can be negative), True for channels
# a0[0]   = direct intercept value (starts near 0)
# a0[1:]  = log-intensity for channels: to start at λ₀, set a0 = log(λ₀) / 0.5 = 2·log(λ₀)

INIT_LAMBDA = 0.1    # desired starting multiplicative intensity for channel states
a0 = jnp.concatenate([
    jnp.zeros(1),                                            # intercept: linear space
    jnp.full(n_states - 1, 2.0 * jnp.log(INIT_LAMBDA)),    # channels: log-intensity space
])
P0 = jnp.ones(n_states)

# positivity_idx already defined in cell-2: [0, 1, 1, ..., 1]
# cast to bool for the EKF
positivity_mask = positivity_idx.astype(bool)

# 2-element priors: [level, shared_regressors]
sigma_q_loc_prior = np.array([0.05, 0.01])
sigma_q_scale_prior = np.array([0.05, 0.01])

print("a0 shape:", a0.shape)
print(f"  a0[0] = {a0[0]:.3f}  (intercept, linear space)")
print(f"  a0[1] = {a0[1]:.3f}  (channel, log-intensity → λ₀ = exp(0.5·a0) = {jnp.exp(0.5 * a0[1]):.3f})")
print("P0:", P0)

In [ ]:
# Test run — mixed EKF: intercept linear, channels exp(0.5·a)
sigma_h = jnp.array(0.1)
sigma_q_raw = jnp.array([0.01, 0.01])  # [level, shared_regressors]
sigma_q = jnp.concatenate([sigma_q_raw[:1], jnp.repeat(sigma_q_raw[1:], Z.shape[-1] - 1)])
print("sigma_q expanded:", sigma_q.shape, sigma_q)

lp, at, Pt, _, _, _ = kalman_filter_1d_ekf(
    a0=a0,
    P0=P0,
    sigma_h=sigma_h,
    sigma_q=sigma_q,
    y=y,
    Z=Z,
    logp=True,
    positivity_idx=positivity_mask,
)

# recover intensities: intercept stays as-is, channels via exp(0.5·a)
lam = jnp.where(positivity_mask, jnp.exp(jnp.clip(0.5 * at, -10.0, 10.0)), at)

print("lp:", lp)
print("at shape:", at.shape)
print("intercept at[-1,0]:", at[-1, 0], "  ← linear space")
print("channel lam[-1,1]:", lam[-1, 1], "  ← λ = exp(0.5·a)")

In [ ]:
def _nuts_fn(a0, P0):
    sigma_h = numpyro.sample(
        'sigma_h',
        dist.TruncatedNormal(
            0.1, 1.0, high=1.0, low=1e-5
        )
    )

    # sample 2-element sigma_q: [level, shared_regressors]
    sigma_q_raw = numpyro.sample(
        'sigma_q',
        dist.TruncatedNormal(
            sigma_q_loc_prior,
            sigma_q_scale_prior,
            high=0.1,
            low=1e-5
        )
    )

    # expand to full n_states: [level, reg1, reg2, ..., regN]
    n_regressors = Z.shape[-1] - 1
    sigma_q = jnp.concatenate([sigma_q_raw[:1], jnp.repeat(sigma_q_raw[1:], n_regressors)])

    # intercept is linear; channel states use exp(0.5·a)
    lp, at, _, _, _, _ = kalman_filter_1d_ekf(
        a0=a0,
        P0=P0,
        sigma_h=sigma_h,
        sigma_q=sigma_q,
        y=y,
        Z=Z,
        logp=True,
        positivity_idx=positivity_mask,
        a_obs_loc=a_obs_loc,
        a_obs_var=a_obs_var,
    )

    # intercept: linear (can be negative); channels: λ = exp(0.5·a) > 0
    lam = jnp.where(positivity_mask, jnp.exp(jnp.clip(0.5 * at, -10.0, 10.0)), at)

    numpyro.factor("lp", lp)
    numpyro.deterministic("at", at)
    numpyro.deterministic("lam", lam)
    numpyro.deterministic("mu", jnp.sum(Z * lam, -1))

In [ ]:
nuts_kernel = NUTS(_nuts_fn)
# kernel = HMCGibbs(
#     inner_kernel=nuts_kernel,
#     gibbs_fn=_gibbs_fn,
#     gibbs_sites=["smooth_alpha"],
# )
mcmc = MCMC(
    nuts_kernel, 
    num_warmup=400, 
    num_samples=400,
    num_chains=4,
)
rng_key = random.PRNGKey(0)
rng_keys = random.split(rng_key, 2)
rng_key = rng_keys[0]
mcmc_rng_key = rng_keys[1]
mcmc.run(mcmc_rng_key, a0, P0)

In [ ]:
from numpyro.diagnostics import summary

# print_summary covers R-hat and n_eff for all sites
mcmc.print_summary()

# detailed per-parameter diagnostics for scalar params
samples = mcmc.get_samples(group_by_chain=True)
diag = summary(samples, prob=0.9)

# flag any R-hat > 1.01 or n_eff < 100
# use max(r_hat) / min(n_eff) for multi-dimensional params (at, lam, mu, etc.)
print("\n--- Convergence flags ---")
for param, stats in diag.items():
    rhat = float(np.asarray(stats["r_hat"]).max())
    neff = float(np.asarray(stats["n_eff"]).min())
    if rhat > 1.01 or neff < 100:
        print(f"  WARNING  {param:<20}  r_hat={rhat:.4f}  n_eff={neff:.1f}")

In [ ]:
posteriors_dict = mcmc.get_samples()
posteriors_dict.keys()

In [ ]:
state_labels = ["intercept"] + regressors
coefs_lookup = coefs_df.set_index("regressor")["coef"] if coefs_df is not None else None

obs_dates = df["date"].values[obs_idx] if len(obs_idx) > 0 else []

# lam = exp(0.5·at) — the multiplicative intensities λ_t, always positive
lam_samples = np.array(posteriors_dict["lam"])
lam_lo, lam_mid, lam_hi = np.quantile(lam_samples, q=[0.05, 0.5, 0.95], axis=0)

fig, axes = plt.subplots(len(state_labels), 1, figsize=(12, 2.5 * len(state_labels)), sharex=True)
for i, (ax, label) in enumerate(zip(axes, state_labels)):
    ax.plot(df["date"], lam_mid[:, i], linewidth=0.8)
    ax.fill_between(df["date"], lam_lo[:, i], lam_hi[:, i], alpha=0.25, label="90% CI")
    if i > 0 and coefs_lookup is not None and label in coefs_lookup.index:
        ax.axhline(coefs_lookup[label], color="red", linestyle="--", linewidth=1.0, label="true coef")
    # mark time points where this state was disclosed (finite a_obs_var = active disclosure)
    if USE_DISCLOSURE and len(obs_idx) > 0 and np.any(np.isfinite(np.array(a_obs_var)[obs_idx, i])):
        first = True
        for d in obs_dates:
            ax.axvline(d, color="orange", linestyle=":", linewidth=0.9, alpha=0.8,
                       label="truth disclosed" if first else None)
            first = False
    ax.legend(fontsize=8)
    ax.set_title(f"{label}  [λ = exp(0.5·a), always > 0]", fontsize=9, loc="left")
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("date")
title = "Log-normal EKF — multiplicative intensities λ_t = exp(0.5·a_t)  [disclosure ON]" if USE_DISCLOSURE \
    else "Log-normal EKF — multiplicative intensities λ_t = exp(0.5·a_t)  [disclosure OFF]"
fig.suptitle(title, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
mu_samples = np.array(posteriors_dict["mu"])
eps_samples = np.random.normal(
    0,
    np.array(posteriors_dict["sigma_h"])[:, None],
    size=mu_samples.shape,
)

# compute p5, p50, p95 for forecast
yhat_lower, yhat_mid, yhat_upper = np.quantile(np.exp(mu_samples + eps_samples) * response_norm, q=[0.05, 0.5, 0.95], axis=0)
# yhat_lower, yhat_mid, yhat_upper = np.quantile((mu_samples + eps_samples) * response_std + response_mean, q=[0.05, 0.5, 0.95], axis=0)

In [ ]:
import matplotlib.pyplot as plt
n = 100
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(np.arange(n), response[-n:], label='Observed', alpha=0.5, color="orange", s=10)
ax.plot(np.arange(n), yhat_mid[-n:], label='Forecast', linestyle='--', alpha=0.8, color="dodgerblue")
ax.fill_between(np.arange(n), yhat_lower[-n:], yhat_upper[-n:], alpha=0.5, label='95% Prediction Interval', color="dodgerblue")
ax.set_title('State Space Model Forecast')